# 04 — Parser SOME/IP + Validação

Roda `01_parse_pcap.py` nos 7 PCAPs e valida os resultados contra as métricas
exttraídas pelo tshark (verificadas contra o Wireshark).

**Critério de aprovação:** contagem de frames SOME/IP e SOME/IP-SD por PCAP
deve bater com o ground truth do tshark dentro de ±0.5%.

In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd
import numpy as np

# Caminhos
ROOT       = Path(r'C:\Mestrado\SDV_Research')
SCRIPTS    = ROOT / 'experiments' / 'files'
PCAP_DIR   = ROOT / 'experiments' / 'notebooks' / 'data' / 'pcap'
STATS_FILE = ROOT / 'data_exploration' / 'pcap_analysis' / 'pcap_stats.json'
OUT_CSV    = ROOT / 'data' / 'parsed_packets.csv'

sys.path.insert(0, str(SCRIPTS))

with open(STATS_FILE, encoding='utf-8') as f:
    TSHARK_REF = json.load(f)

print('Referência tshark carregada:')
for label, r in TSHARK_REF.items():
    print(f'  {label:<14} n_si={r["n_si"]:>10,}  n_sd={r["n_sd"]:>7,}')

## 1. Teste rápido — benigno

Roda o parser em `benign_traffic.pcap` e compara contra o tshark.
Se passar, roda todos os PCAPs.

In [ ]:
from importlib import import_module
import importlib

# Importa o parser
parser_mod = importlib.import_module('01_parse_pcap'.replace('-','_'))
# Python não importa módulos que começam com número — usar importlib.util
import importlib.util
spec = importlib.util.spec_from_file_location('parse_pcap', SCRIPTS / '01_parse_pcap.py')
parse_pcap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parse_pcap)
print('Módulo carregado:', parse_pcap.__file__)

In [ ]:
import time

OUT_BENIGN = ROOT / 'data' / 'parsed_benign_test.csv'

print('Rodando parser em benign_traffic.pcap ...')
t0 = time.time()
parse_pcap.process_all_pcaps(
    pcap_dir=str(PCAP_DIR),
    output_csv=str(OUT_BENIGN),
    pcap_filter=['benign_traffic.pcap'],   # só benigno
)
elapsed = time.time() - t0
print(f'\nConcluído em {elapsed:.0f}s')

## 2. Validação — benigno vs tshark

In [ ]:
df = pd.read_csv(OUT_BENIGN, low_memory=False)

ref = TSHARK_REF['benign']

parsed_si  = int(df['someip_valid'].sum())        # frames com SOME/IP válido
parsed_sd  = int(df['is_sd'].fillna(False).sum()) # frames SOME/IP-SD

# service ID distribution
parsed_svc = (df[df['someip_valid'] == True]
              .assign(svc_hex=lambda x: x['service_id'].apply(
                  lambda v: f'0x{int(v):04x}' if pd.notna(v) else None))
              ['svc_hex'].value_counts().to_dict())

print('=' * 60)
print('VALIDAÇÃO — benign_traffic.pcap')
print('=' * 60)
print(f'\n{"Métrica":<28} {"tshark":>12} {"parser":>12} {"diff":>8} {"OK?"}')
print('-' * 65)

def check(name, ref_val, parsed_val, tol=0.005):
    diff = parsed_val - ref_val
    pct  = abs(diff) / ref_val * 100 if ref_val else 0
    ok   = pct <= tol * 100
    mark = 'OK' if ok else 'FALHOU'
    print(f'{name:<28} {ref_val:>12,} {parsed_val:>12,} {diff:>+8,} {mark}')
    return ok

results = []
results.append(check('SOME/IP frames',  ref['n_si'], parsed_si))
results.append(check('SOME/IP-SD frames', ref['n_sd'], parsed_sd))

print()
print('Service IDs — tshark vs parser:')
all_svcs = set(ref['svc_counts']) | set(parsed_svc)
for svc in sorted(all_svcs):
    rv = ref['svc_counts'].get(svc, 0)
    pv = parsed_svc.get(svc, 0)
    diff = pv - rv
    mark = 'OK' if abs(diff) / max(rv, 1) < 0.01 else 'DIFF'
    print(f'  {svc:<8}  tshark={rv:>10,}  parser={pv:>10,}  {diff:>+8,}  {mark}')

print()
all_ok = all(results)
print(f'\nRESULTADO: {"APROVADO" if all_ok else "REPROVADO"}')
if not all_ok:
    print('  Investigar diferença antes de rodar todos os PCAPs.')

## 3. Rodar todos os PCAPs

Só executar após o benigno ser aprovado.

In [ ]:
import time

print('Rodando parser em todos os 7 PCAPs...')
print('(pode levar 20-40 min dependendo da máquina)\n')

t0 = time.time()
parse_pcap.process_all_pcaps(
    pcap_dir=str(PCAP_DIR),
    output_csv=str(OUT_CSV),
)
elapsed = time.time() - t0
print(f'\nConcluído em {elapsed/60:.1f} min')
print(f'CSV salvo em: {OUT_CSV}')

## 4. Validação completa — todos os PCAPs

In [ ]:
df_all = pd.read_csv(OUT_CSV, low_memory=False)
print(f'Total de linhas no CSV: {len(df_all):,}')
print(f'Colunas: {list(df_all.columns)}\n')

LABEL_MAP = {
    'benign_traffic.pcap':              'benign',
    'dos_noti_flood.pcap':              'dos_noti',
    'fuzzy_sd_offer_rand_noti(1).pcap': 'fuzzy(1)',
    'fuzzy_sd_offer_rand_noti(2).pcap': 'fuzzy(2)',
    'fuzzy_sd_offer_rand_noti(3).pcap': 'fuzzy(3)',
    'mitm_multi_attacker.pcap':         'mitm_multi',
    'mitm_single_attacker.pcap':        'mitm_single',
}

print(f'{"PCAP":<14} {"tshark n_si":>12} {"parser n_si":>12} {"diff":>8} '
      f'{"tshark n_sd":>12} {"parser n_sd":>12} {"diff":>8} {"OK?"}')
print('-' * 100)

all_ok = True
for pcap_file, ref_label in LABEL_MAP.items():
    sub = df_all[df_all['pcap_file'] == pcap_file]
    p_si = int(sub['someip_valid'].sum())
    p_sd = int(sub['is_sd'].fillna(False).sum())

    ref = TSHARK_REF.get(ref_label, {})
    r_si = ref.get('n_si', 0)
    r_sd = ref.get('n_sd', 0)

    d_si = p_si - r_si
    d_sd = p_sd - r_sd
    ok_si = abs(d_si) / max(r_si, 1) < 0.005
    ok_sd = abs(d_sd) / max(r_sd, 1) < 0.01
    ok = ok_si and ok_sd
    all_ok = all_ok and ok
    mark = 'OK' if ok else 'DIFF'

    print(f'{ref_label:<14} {r_si:>12,} {p_si:>12,} {d_si:>+8,} '
          f'{r_sd:>12,} {p_sd:>12,} {d_sd:>+8,} {mark}')

print()
print(f'RESULTADO GLOBAL: {"APROVADO" if all_ok else "VERIFICAR DIFFS"}')

## 5. Inspeção das colunas do CSV

In [ ]:
si = df_all[df_all['someip_valid'] == True]

print(f'Frames SOME/IP válidos: {len(si):,}')
print(f'  com transport_payload_hex : {si["transport_payload_hex"].notna().sum():,}')
print(f'  com someip_payload_hex    : {si["someip_payload_hex"].notna().sum():,}')
print()
print('Amostra de transport_payload_hex (primeiros 5):')
print(si['transport_payload_hex'].dropna().head())
print()
print('Amostra de someip_payload_hex (primeiros 5):')
print(si['someip_payload_hex'].dropna().head())
print()
print('Distribuição de msg_type:')
print(si['msg_type'].value_counts().head(10))
print()
print('Distribuição de service_id (hex):')
print(si['service_id'].apply(lambda v: f'0x{int(v):04x}' if pd.notna(v) else None)
      .value_counts().head(10))